# Retrieve the Dataset

In [8]:
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")  # suppress all warnings

# Use only numeric features (age, education-num, capital-gain, capital-loss, hours-per-week)
x_train = np.loadtxt("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
                        usecols=(0, 4, 10, 11, 12), delimiter=",")

y_train = np.loadtxt("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
                        usecols=14, dtype=str, delimiter=",")


x_test = np.loadtxt("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test",
                        usecols=(0, 4, 10, 11, 12), delimiter=",", skiprows=1)

y_test = np.loadtxt("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test",
                        usecols=14, dtype=str, delimiter=",", skiprows=1)

# Trim trailing period "." from label
y_test = np.array([a[:-1] for a in y_test])

y_train[y_train == ' <=50K'] = 0
y_train[y_train == ' >50K'] = 1
y_train = y_train.astype(int)

y_test[y_test == ' <=50K'] = 0
y_test[y_test == ' >50K'] = 1
y_test = y_test.astype(int)

#give the columns labels
column_labels = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

#attach the column labels to the data
df_X_train = pd.DataFrame(x_train, columns=column_labels)
df_X_test = pd.DataFrame(x_test, columns=column_labels)

df_y_train = pd.DataFrame(y_train, columns=['income'])
df_y_test = pd.DataFrame(y_test, columns=['income'])



# Train a model to obtain the baseline

In [9]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

from apt.utils.datasets import ArrayDataset
from apt.utils.models import SklearnClassifier, ModelOutputType
from sklearn.tree import DecisionTreeClassifier

base_est = DecisionTreeClassifier()
model = SklearnClassifier(base_est, ModelOutputType.BINARY)
model.fit(ArrayDataset(x_train, y_train))

print('Base model accuracy: ', model.score(ArrayDataset(df_X_test, df_y_test)))

Base model accuracy:  0.8188686198636448


# Get the model's predctions

In [10]:
from apt.minimization import GeneralizeToRepresentative
from sklearn.model_selection import train_test_split

X_generalizer_train, x_test, y_generalizer_train, y_test = train_test_split(df_X_test, df_y_test, stratify=df_y_test,
                                                                test_size = 0.4, random_state = 38)

x_train_predictions = model.predict(ArrayDataset(X_generalizer_train))
if x_train_predictions.shape[1] > 1:
    x_train_predictions = np.argmax(x_train_predictions, axis=1)

In [11]:
from apt.minimization.minimizer import *
# We allow a 10% deviation in accuracy from the original model accuracy
minimizer = GeneralizeToRepresentative(model, target_accuracy=0.90)

minimizer.fit(dataset=ArrayDataset(X_generalizer_train, x_train_predictions))
transformed = minimizer.transform(dataset=ArrayDataset(x_test))
print('Accuracy on minimized data: ', model.score(test_data=ArrayDataset(transformed, y_test)))
generalizations = minimizer.generalizations
print(generalizations)

Initial accuracy of model on generalized data, relative to original model predictions (base generalization derived from tree, before improvements): 0.921699
Improving generalizations
Pruned tree to level: 1, new relative accuracy: 0.921443
Pruned tree to level: 2, new relative accuracy: 0.920676
Pruned tree to level: 3, new relative accuracy: 0.920676
Pruned tree to level: 4, new relative accuracy: 0.920676
Pruned tree to level: 5, new relative accuracy: 0.920676
Pruned tree to level: 6, new relative accuracy: 0.920420
Pruned tree to level: 7, new relative accuracy: 0.919652
Pruned tree to level: 8, new relative accuracy: 0.919908
Pruned tree to level: 9, new relative accuracy: 0.919396
Pruned tree to level: 10, new relative accuracy: 0.919908
Pruned tree to level: 11, new relative accuracy: 0.919652
Pruned tree to level: 12, new relative accuracy: 0.918117
Pruned tree to level: 13, new relative accuracy: 0.917605
Pruned tree to level: 14, new relative accuracy: 0.913767
Pruned tree to

# K-anonymity tests

In [12]:
from apt.minimization.minimiser_expansion import *
temp = has_k_anonymity(transformed, target_k=2)
temp2 = k_anonymity(transformed)
print(f"has 2-anonymity: {temp}")
print(f"has {temp2}-anonymity")

has 2-anonymity: False
has 1-anonymity


In [13]:
# We test for k-anonymity with k=2
t_k = 2
minimizer2 = GeneralizeToRepresentative(model, mode=K_ANONYMITY_MODE, target_k=t_k)

minimizer2.fit(dataset=ArrayDataset(X_generalizer_train, x_train_predictions))
transformed2 = minimizer2.transform(dataset=ArrayDataset(x_test))
print('Accuracy on minimized data: ', model.score(test_data=ArrayDataset(transformed2, y_test)))
generalizations2 = minimizer2.generalizations
print(generalizations2)
temp = has_k_anonymity(transformed2, target_k=t_k)
temp2 = k_anonymity(transformed2)
print(f"has {t_k}-anonymity: {temp}")
print(f"has {temp2}-anonymity")


Initial accuracy of model on generalized data, relative to original model predictions (base generalization derived from tree, before improvements): 0.921699
K anonymity is: 1.000000
Target K anonymity is: 2.000000
Improving generalizations
Pruned tree to level: 1, new k anonymity: 1.000000
Pruned tree to level: 2, new k anonymity: 1.000000
Pruned tree to level: 3, new k anonymity: 1.000000
Pruned tree to level: 4, new k anonymity: 1.000000
Pruned tree to level: 5, new k anonymity: 1.000000
Pruned tree to level: 6, new k anonymity: 1.000000
Pruned tree to level: 7, new k anonymity: 1.000000
Pruned tree to level: 8, new k anonymity: 1.000000
Pruned tree to level: 9, new k anonymity: 1.000000
Pruned tree to level: 10, new k anonymity: 1.000000
Pruned tree to level: 11, new k anonymity: 1.000000
Pruned tree to level: 12, new k anonymity: 1.000000
Pruned tree to level: 13, new k anonymity: 1.000000
Pruned tree to level: 14, new k anonymity: 1.000000
Pruned tree to level: 15, new k anonymity

In [14]:
print(transformed2.value_counts())

age   education-num  capital-gain  capital-loss  hours-per-week
35.0  9.0            0.0           0.0           40.0              3519
37.0  9.0            0.0           0.0           50.0              1121
31.0  13.0           0.0           0.0           40.0               523
43.0  13.0           0.0           0.0           50.0               464
51.0  13.0           0.0           0.0           40.0               345
46.0  9.0            14344.0       0.0           40.0               260
28.0  13.0           0.0           0.0           50.0               139
43.0  9.0            0.0           1902.0        40.0                56
34.0  10.0           0.0           2057.0        35.0                35
43.0  10.0           5178.0        0.0           40.0                23
      14.0           6497.0        0.0           50.0                12
65.0  9.0            6418.0        0.0           45.0                 8
19.0  6.0            34095.0       0.0           24.0                 5
